# Stage 3 — MAPPO (Multi-Agent PPO)

Three robots coordinate order assignment in a shared 10×10 warehouse.

## Centralised Training, Decentralised Execution (CTDE)

| Component           | Input              | Output          | Used at              |
|---------------------|--------------------|-----------------|----------------------|
| Shared Actor        | 9-dim local obs    | 3-action logits | Training + Execution |
| Centralised Critic  | 27-dim global state | scalar value   | Training only        |

## Action space (per robot)

| Action    | Behaviour                              | Reward          |
|-----------|----------------------------------------|-----------------|
| 0 Accept  | Navigate to pickup → dropoff           | +100 / −80 / −10|
| 1 Idle    | Idle briefly                           | −20 to −17.5    |
| 2 GoCharge| Navigate to charger, charge to 100%    | −25             |

## Dispatch mechanism
Each order is offered to the **K=2 nearest robots** (by BFS distance).  
If both bid Accept, the one with higher `battery − trip_cost` wins. The loser falls back to Idle.  
Non-eligible robots can only Idle or GoCharge.

**Warm-start**: actor weights copied from Stage-2 AssignmentDQN.  
**Resume**: if `mappo_actor_ckpt.pt` exists in `checkpoints/`, training resumes from it automatically.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import torch
import numpy as np
import json

from envs.marl_env import MultiAgentWarehouse, N_AGENTS, N_ORDERS, OBS_DIM, GLOBAL_DIM
from agents.mappo import AssignmentActor, CentralisedCritic
from utils.plotting import plot_mappo_history
from training.train_mappo import (
    load_nav_policy, load_assign_dqn,
    train_mappo, evaluate_mappo,
    DEVICE, CKPT_DIR
)

print(f'Device      : {DEVICE}')
print(f'Agents      : {N_AGENTS}')
print(f'Orders/ep   : {N_ORDERS}')
print(f'Obs dim     : {OBS_DIM}  (8 base + is_eligible flag)')
print(f'Global dim  : {GLOBAL_DIM}  ({N_AGENTS} × {OBS_DIM})')
print(f'Checkpoints : {CKPT_DIR}')

In [ ]:
nav_model  = load_nav_policy()
assign_dqn = load_assign_dqn()

In [ ]:
ckpt_actor  = os.path.join(CKPT_DIR, 'mappo_actor_ckpt.pt')
ckpt_critic = os.path.join(CKPT_DIR, 'mappo_critic_ckpt.pt')
resume_actor  = ckpt_actor  if os.path.exists(ckpt_actor)  else None
resume_critic = ckpt_critic if os.path.exists(ckpt_critic) else None

if resume_actor:
    print(f'Resuming from checkpoint: {ckpt_actor}')
else:
    print('No checkpoint found — warm-starting from Stage-2 DQN weights')

actor, critic, history = train_mappo(
    nav_model     = nav_model,
    assign_dqn    = assign_dqn,
    episodes      = 10_000,
    n_rollout     = 16,
    clip_eps      = 0.2,
    ppo_epochs    = 4,
    gamma         = 0.99,
    lr_actor      = 5e-5,
    lr_critic     = 1e-4,
    entropy_coef  = 0.02,
    verbose_every = 500,
    resume_actor  = resume_actor,
    resume_critic = resume_critic,
)
print('Training complete.')

In [ ]:
plot_mappo_history(history, out_dir=CKPT_DIR)

In [ ]:
eval_metrics = evaluate_mappo(nav_model, actor, n_eval=200)

In [ ]:
s2_path = os.path.join(CKPT_DIR, 'assign_eval.json')
if os.path.exists(s2_path):
    with open(s2_path) as f:
        s2 = json.load(f)
    print(f'{"Metric":<30}  {"Stage 2":>10}  {"Stage 3":>10}')
    print('-' * 54)
    for key in ['accept_rate', 'charge_rate', 'idle_rate']:
        print(f'  {key:<28}  {s2[key]:>10.3f}  {eval_metrics.get(key, float("nan")):>10.3f}')
    print(f'  {"deliveries/episode":<28}  {s2["deliveries"]:>10.3f}  {eval_metrics["deliveries"]:>10.3f}')
    print(f'  {"team_reward":<28}  {"—":>10}  {eval_metrics["team_reward"]:>10.1f}')
else:
    for k, v in eval_metrics.items():
        print(f'  {k:<30}: {v:.4f}')

In [ ]:
torch.save(actor.state_dict(),  os.path.join(CKPT_DIR, 'mappo_actor.pt'))
torch.save(critic.state_dict(), os.path.join(CKPT_DIR, 'mappo_critic.pt'))

for key, arr in history.items():
    np.save(os.path.join(CKPT_DIR, f'mappo_{key}.npy'), np.array(arr))

with open(os.path.join(CKPT_DIR, 'mappo_eval.json'), 'w') as f:
    json.dump({k: float(v) for k, v in eval_metrics.items()}, f, indent=2)

print(f'Saved: mappo_actor.pt, mappo_critic.pt, mappo_eval.json, mappo_*.npy')
print(f'Location: {CKPT_DIR}')